---
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Week 12 Lab: Neural Network Interpretation & Explainable AI</font></div>
---

Throughout this course you have learned to build, train, optimise, and fine-tune models. But an important question remains:

**How do we know *why* a model made a prediction?**

A highly accurate model is not necessarily a trustworthy model. Consider:

| Scenario | Prediction | The stakeholder asks... |
|---|---|---|
| Medical diagnosis | Cancer (99% confidence) | *"Why? What features indicate cancer?"* |
| Loan approval | Rejected | *"Why was I rejected? Is this fair?"* |
| Autonomous vehicle | Brake | *"Why did it brake? Was it correct?"* |

Modern AI systems increasingly require **explainability**, **transparency**, and **accountability**. This week explores methods for understanding what neural networks learn and how they make decisions.

The lab has two parts: **Part I** covers vision model explanations (Grad-CAM), **Part II** covers NLP model explanations (attention visualisation and counterfactuals).

## 🎯 Learning Objectives

By the end of this lab, you should be able to:

- Explain why interpretability and explainability are important in deep learning
- Distinguish between model performance and model understanding
- Generate visual explanations for CNN image classifiers using Grad-CAM
- Analyse what image regions influence model predictions
- Use transformer attention visualisation to investigate NLP models
- Perform counterfactual analysis to probe model behaviour
- Identify potential biases and spurious correlations in trained models
- Critically evaluate whether an explanation is trustworthy

**Prerequisites:** Weeks 3–11 (CNNs, transfer learning, transformers, foundation models, representation learning)

**Duration:** ~2–3 hours

---

## Setup

In [ ]:
!pip install -q grad-cam transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import requests
from io import BytesIO
import os, warnings

from torchvision import models, transforms

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore', message='.*HF_TOKEN.*')
warnings.filterwarnings('ignore', message='.*unauthenticated.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 0: Interpretability vs Explainability 

| Concept | Description | Example |
|---|---|---|
| **Interpretability** | Understanding how a model works internally | Inspecting learned filters in a CNN |
| **Explainability** | Explaining why a specific prediction was made | Highlighting which pixels influenced a "cat" prediction |
| **Debugging** | Identifying model failures and their causes | Finding that a model relies on background, not object |
| **Trustworthiness** | Determining whether predictions should be relied upon | Deciding if a model is safe for clinical deployment |

### Discussion

Consider two models:
- **Model A:** 98% accuracy, no explanation available
- **Model B:** 96% accuracy, provides Grad-CAM explanations showing it focuses on the correct anatomical regions

Which would you deploy in a hospital? Why?

### 🤔 Questions

1. Is accuracy alone sufficient for deployment in high-stakes applications?
2. Why might explainability be especially important in healthcare, finance, and criminal justice?
3. Can a model be accurate but untrustworthy? Give an example.

*Double-click here to write your answers...*

---
# Part I: Vision Model Explanations

---

# Part 1: Image Classification with a Pretrained CNN 

We use a pretrained ResNet-18 (from ImageNet) to classify images, then investigate **why** it makes its predictions.

## Task 1.1 — Load a Pretrained ResNet and Sample Images

In [ ]:
# Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights).to(device)
resnet.eval()

# ImageNet preprocessing
preprocess = weights.transforms()

# Load ImageNet class labels
IMAGENET_CLASSES = weights.meta['categories']
print(f'Model loaded. {len(IMAGENET_CLASSES)} ImageNet classes.')

In [ ]:
# Use sample images from torchvision's built-in test images
from torchvision.datasets import ImageNet
import urllib.request

# Reliable image URLs (Unsplash — free, stable links)
image_urls = {
    'dog': 'https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=640',
    'cat': 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=640',
    'bird': 'https://images.unsplash.com/photo-1444464666168-49d633b86797?w=640',
}

images = {}
for name, url in image_urls.items():
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req, timeout=10)
        img = Image.open(BytesIO(response.read())).convert('RGB')
        images[name] = img
        print(f'  Loaded: {name} ({img.size})')
    except Exception as e:
        print(f'  Failed: {name} ({e})')

# Fallback: if downloads fail, use a generated test image
if len(images) == 0:
    print('\nAll downloads failed. Using FashionMNIST as fallback.')
    print('You can also upload your own images using the file browser (left panel).')

In [ ]:

images = {}
for name, url in image_urls.items():
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        images[name] = img
        print(f'  Loaded: {name} ({img.size})')
    except:
        print(f'  Failed to load: {name}')

# Display
fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Task 1.2 — Run Inference and Show Top-5 Predictions

In [ ]:
def predict_top5(model, image, preprocess, class_names):
    """Run inference and return top-5 predictions."""
    input_tensor = preprocess(image).unsqueeze(0).to(device)
    
    with torch.inference_mode():
        logits = model(input_tensor)
        probs = F.softmax(logits, dim=1)[0]
    
    top5_probs, top5_indices = probs.topk(5)
    results = [(class_names[idx.item()], prob.item()) for idx, prob in zip(top5_indices, top5_probs)]
    return results

# Predict for each image
for name, img in images.items():
    top5 = predict_top5(resnet, img, preprocess, IMAGENET_CLASSES)
    print(f'\n{name.upper()}:')
    for i, (cls, prob) in enumerate(top5):
        bar = '█' * int(prob * 40)
        print(f'  {i+1}. {cls:<30} {prob:.2%} {bar}')

### 🤔 Questions

1. Were the predictions correct?
2. What other classes appeared in the top-5? Are they related?
3. How confident was the model? Does confidence always mean correctness?

*Double-click here to write your answers...*

---
# Part 2: Grad-CAM — Where Is the Model Looking? 

## What Is Grad-CAM?

Grad-CAM (Gradient-weighted Class Activation Mapping) answers:

**Which image regions contributed most to a prediction?**

The method:
1. Compute the gradient of the predicted class score with respect to the feature maps of a target convolutional layer
2. Global average pool the gradients to get per-channel importance weights
3. Compute a weighted sum of feature maps → produces a heatmap
4. Apply ReLU (keep only positive contributions) and resize to input image dimensions

The result: a heatmap showing which spatial regions the model relied on.


## Task 2.1 — Generate Grad-CAM Heatmaps

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target layer: last convolutional layer of ResNet-18
target_layer = resnet.layer4[-1]

cam = GradCAM(model=resnet, target_layers=[target_layer])
print(f'Grad-CAM target layer: {target_layer.__class__.__name__}')

In [ ]:
def generate_gradcam(cam_model, image, preprocess_fn, class_names, target_class=None):
    """Generate Grad-CAM heatmap for an image."""
    # Preprocess for model
    input_tensor = preprocess_fn(image).unsqueeze(0).to(device)
    
    # Get prediction
    with torch.inference_mode():
        logits = resnet(input_tensor)
        pred_class = logits.argmax(dim=1).item()
    
    # Use predicted class or specified class
    if target_class is None:
        target_class = pred_class
    
    targets = [ClassifierOutputTarget(target_class)]
    
    # Generate heatmap
    grayscale_cam = cam_model(input_tensor=input_tensor, targets=targets)[0]
    
    # Prepare image for overlay (resize to 224x224 and normalise to [0,1])
    img_resized = image.resize((224, 224))
    img_array = np.array(img_resized).astype(np.float32) / 255.0
    
    # Create overlay
    overlay = show_cam_on_image(img_array, grayscale_cam, use_rgb=True)
    
    return grayscale_cam, overlay, class_names[pred_class], class_names[target_class]

In [ ]:
# Generate Grad-CAM for all images
fig, axes = plt.subplots(len(images), 3, figsize=(12, 4 * len(images)))

for row, (name, img) in enumerate(images.items()):
    heatmap, overlay, pred_name, target_name = generate_gradcam(
        cam, img, preprocess, IMAGENET_CLASSES
    )
    
    # Original
    axes[row, 0].imshow(img.resize((224, 224)))
    axes[row, 0].set_title(f'Original ({name})', fontsize=10)
    axes[row, 0].axis('off')
    
    # Heatmap
    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 1].set_title(f'Grad-CAM Heatmap', fontsize=10)
    axes[row, 1].axis('off')
    
    # Overlay
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Overlay → "{pred_name}"', fontsize=10)
    axes[row, 2].axis('off')

plt.suptitle('Grad-CAM Visualisation', fontsize=14)
plt.tight_layout()
plt.show()

### 🤔 Questions

1. What image regions were highlighted as important?
2. Does the explanation seem reasonable — is the model focusing on the object itself?
3. Is the model focusing on the object or the background? What would it mean if the model relied on background features?
4. Could you trust a model that correctly predicts "dog" but focuses on grass instead of the dog?

*Double-click here to write your answers...*

---
# Part 3: Detecting Bias and Spurious Correlations

A model can make **correct predictions for the wrong reasons**.

### The Classic Example

A wolf vs husky classifier achieved high accuracy — but Grad-CAM revealed it had learned to detect **snow in the background** (wolves appear in snow, huskies on grass), not the animals themselves.

Let's investigate whether our model uses the object or the context.

## Task 3.1 — Grad-CAM for Different Target Classes

In [ ]:
# What does the model look at when predicting DIFFERENT classes for the SAME image?
# Take the dog image and ask: what regions support "dog" vs "grass" vs "tennis ball"?

test_image = list(images.values())[0]  # dog image
test_name = list(images.keys())[0]

# Find class indices for interesting target classes
target_classes = {}
search_terms = ['labrador', 'golden retriever', 'tennis ball', 'lawn_mower', 'park bench']
for i, cls in enumerate(IMAGENET_CLASSES):
    for term in search_terms:
        if term.lower() in cls.lower():
            target_classes[cls] = i

# Also add the predicted class
with torch.inference_mode():
    pred_idx = resnet(preprocess(test_image).unsqueeze(0).to(device)).argmax().item()
target_classes[f'{IMAGENET_CLASSES[pred_idx]} (predicted)'] = pred_idx

print('Target classes to investigate:')
for cls, idx in target_classes.items():
    print(f'  {idx}: {cls}')

In [ ]:
# Generate Grad-CAM for each target class
n_targets = min(4, len(target_classes))
fig, axes = plt.subplots(1, n_targets + 1, figsize=(4 * (n_targets + 1), 4))

# Original image
axes[0].imshow(test_image.resize((224, 224)))
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for i, (cls_name, cls_idx) in enumerate(list(target_classes.items())[:n_targets]):
    heatmap, overlay, _, _ = generate_gradcam(
        cam, test_image, preprocess, IMAGENET_CLASSES, target_class=cls_idx
    )
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f'Target: {cls_name}', fontsize=8)
    axes[i + 1].axis('off')

plt.suptitle('Same Image, Different Target Classes — Where Does the Model Look?', fontsize=12)
plt.tight_layout()
plt.show()

### 🤔 Questions 

1. Does the model focus on different regions for different target classes?
2. Can a model be correct for the wrong reason? How would Grad-CAM reveal this?
3. How might Grad-CAM help identify dataset problems during model development?
4. If you discovered that your model relied on background features, what would you do to fix it?

*Double-click here to write your answers...*

---
# Part 4: Comparing CNN Architectures 

Do different architectures "see" differently? Let's compare ResNet-18 and EfficientNet-B0 on the same images.

In [ ]:
# Load EfficientNet-B0
effnet_weights = models.EfficientNet_B0_Weights.DEFAULT
effnet = models.efficientnet_b0(weights=effnet_weights).to(device)
effnet.eval()
effnet_preprocess = effnet_weights.transforms()

# Grad-CAM for EfficientNet (target: last conv layer in features)
effnet_target_layer = effnet.features[-1]
effnet_cam = GradCAM(model=effnet, target_layers=[effnet_target_layer])

print('EfficientNet-B0 loaded.')

In [ ]:
# Compare both models on the same image
test_img = list(images.values())[0]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for row, (model_obj, cam_obj, prep, name) in enumerate([
    (resnet, cam, preprocess, 'ResNet-18'),
    (effnet, effnet_cam, effnet_preprocess, 'EfficientNet-B0'),
]):
    # Predict
    input_t = prep(test_img).unsqueeze(0).to(device)
    with torch.inference_mode():
        logits = model_obj(input_t)
        probs = F.softmax(logits, dim=1)[0]
        pred_idx = logits.argmax().item()
        pred_prob = probs[pred_idx].item()
    
    # Grad-CAM
    targets = [ClassifierOutputTarget(pred_idx)]
    heatmap = cam_obj(input_tensor=input_t, targets=targets)[0]
    img_arr = np.array(test_img.resize((224, 224))).astype(np.float32) / 255.0
    overlay = show_cam_on_image(img_arr, heatmap, use_rgb=True)
    
    axes[row, 0].imshow(test_img.resize((224, 224)))
    axes[row, 0].set_title(f'{name}', fontsize=10)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 1].set_title(f'Heatmap', fontsize=10)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'"{IMAGENET_CLASSES[pred_idx]}" ({pred_prob:.0%})', fontsize=10)
    axes[row, 2].axis('off')

plt.suptitle('ResNet-18 vs EfficientNet-B0', fontsize=13)
plt.tight_layout()
plt.show()

### 🤔 Questions

1. Do the two architectures focus on the same image regions?
2. Which model appears more focused on the actual object?
3. If explanations disagree between models, which should you trust?

*Double-click here to write your answers...*

---
# Part II: NLP Model Explanations

---

# Part 5: Transformer Sentiment Analysis 

In Week 8 you used pretrained transformers for classification. Now we investigate **why** a transformer makes its predictions, using attention visualisation and counterfactual analysis.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
nlp_tokenizer = AutoTokenizer.from_pretrained(model_name)
nlp_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, token=False, attn_implementation='eager'
).to(device)
nlp_model.eval()

SENTIMENT_LABELS = ['NEGATIVE', 'POSITIVE']
print(f'Loaded: {model_name}')

In [ ]:
def predict_sentiment(model, tokenizer, text):
    """Predict sentiment with confidence."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.inference_mode():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)[0]
    pred_idx = probs.argmax().item()
    return SENTIMENT_LABELS[pred_idx], probs[pred_idx].item(), probs.cpu().numpy()

test_texts = [
    "I absolutely loved this movie.",
    "This was a complete waste of time.",
    "The acting was excellent but the story was terrible.",
    "I am not unhappy with the result.",
    "The food was okay, nothing special.",
]

for text in test_texts:
    label, conf, probs = predict_sentiment(nlp_model, nlp_tokenizer, text)
    print(f'  "{text}"')
    print(f'  → {label} ({conf:.1%}) | NEG: {probs[0]:.1%}, POS: {probs[1]:.1%}\n')

### 🤔 Questions 

1. Were all predictions reasonable?
2. Which example was hardest for the model? Why?
3. How does the model handle the double negation ("not unhappy")?
4. Is high confidence always a sign of correctness?

*Double-click here to write your answers...*

---
# Part 6: Attention Visualisation

Transformers compute **attention weights** that indicate how much each token "attends to" every other token. By visualising these weights, we can see which words the model considers most important for its prediction.

In [ ]:
def get_attention_weights(model, tokenizer, text):
    """Extract attention weights from the model."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    
    with torch.inference_mode():
        outputs = model(**inputs, output_attentions=True)
    
    # attentions: tuple of (num_layers,) each with shape (batch, num_heads, seq_len, seq_len)
    attentions = outputs.attentions
    logits = outputs.logits
    probs = F.softmax(logits, dim=1)[0]
    pred_idx = probs.argmax().item()
    
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    return attentions, tokens, SENTIMENT_LABELS[pred_idx], probs[pred_idx].item()

In [ ]:
def plot_token_importance(attentions, tokens, title='', layer=-1):
    """
    Compute per-token importance by averaging attention TO each token
    across all heads in a given layer, then taking the [CLS] token's attention.
    """
    # Use attention from [CLS] token (index 0) in the specified layer
    # Shape: (num_heads, seq_len)
    cls_attention = attentions[layer][0, :, 0, :].cpu().numpy()  # all heads, CLS row
    
    # Average across heads
    avg_attention = cls_attention.mean(axis=0)  # (seq_len,)
    
    # Normalise
    avg_attention = avg_attention / avg_attention.max()
    
    # Plot
    fig, ax = plt.subplots(figsize=(max(8, len(tokens) * 0.8), 2))
    colors = plt.cm.Reds(avg_attention)
    bars = ax.bar(range(len(tokens)), avg_attention, color=colors, edgecolor='white')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Attention Weight')
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualise attention for each test sentence
for text in test_texts[:3]:  # positive, negative, mixed
    attentions, tokens, label, conf = get_attention_weights(nlp_model, nlp_tokenizer, text)
    plot_token_importance(
        attentions, tokens,
        title=f'"{text}" → {label} ({conf:.0%})',
        layer=-1  # last layer
    )

## Task 6.2 — Attention Across Layers

In [ ]:
# Compare attention in early vs late layers
text = "The acting was excellent but the story was terrible."
attentions, tokens, label, conf = get_attention_weights(nlp_model, nlp_tokenizer, text)

print(f'DistilBERT has {len(attentions)} layers\n')

for layer_idx in [0, 2, -1]:
    layer_name = f'Layer {layer_idx}' if layer_idx >= 0 else f'Layer {len(attentions) + layer_idx} (last)'
    plot_token_importance(
        attentions, tokens,
        title=f'{layer_name}: "{text[:40]}..."',
        layer=layer_idx
    )

### 🤔 Questions 

1. Which words receive the highest attention in each sentence?
2. Does the attention align with your intuition about which words carry sentiment?
3. Do attention patterns differ between early and late layers? Why?
4. For the mixed-sentiment sentence, does attention highlight both "excellent" and "terrible"?

*Double-click here to write your answers...*

---
# Part 7: Is Attention an Explanation? 

Attention visualisation is useful, but there's an important caveat:

> *"Attention is not Explanation."* — Jain & Wallace, NAACL 2019

A highly attended word may not be the **true cause** of a prediction. Consider:

```
"I absolutely loved this movie."
```

If attention focuses heavily on "movie," does that prove "movie" caused the positive prediction? No — the model may attend to "movie" to determine the **topic**, while the sentiment signal comes from "loved" through a different mechanism.

### Limitations of Attention as Explanation

- Attention shows **correlation** (what the model looks at), not **causation** (what drives the prediction)
- Different attention heads may tell contradictory stories
- Averaging across heads can obscure important patterns
- High attention ≠ high importance for the output

### 🤔 Questions

1. Do you agree that "attention is not explanation"? Why or why not?
2. If attention doesn't explain causation, what does it show?
3. How could you test whether a word truly *causes* a prediction? (Hint: the next section)

*Double-click here to write your answers...*

---
# Part 8: Counterfactual Explanations — What Would Change the Prediction? 

A more rigorous way to understand a model is to ask:

> **What minimal change to the input would change the prediction?**

This is a **counterfactual explanation** — it identifies which words are truly *causal* by observing what happens when they change.

In [ ]:
def counterfactual_analysis(model, tokenizer, original, modified):
    """Compare predictions between original and modified text."""
    orig_label, orig_conf, orig_probs = predict_sentiment(model, tokenizer, original)
    mod_label, mod_conf, mod_probs = predict_sentiment(model, tokenizer, modified)
    
    print(f'  Original: "{original}"')
    print(f'    → {orig_label} ({orig_conf:.1%})')
    print(f'  Modified: "{modified}"')
    print(f'    → {mod_label} ({mod_conf:.1%})')
    
    changed = '✅ FLIPPED' if orig_label != mod_label else '— same prediction'
    print(f'  Result: {changed}\n')

In [ ]:
print('=== Counterfactual Experiments ===\n')

print('--- Single word swaps ---')
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "I loved this movie.", "I hated this movie.")
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "The food was excellent.", "The food was terrible.")
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "This is a good product.", "This is a bad product.")

print('--- Negation ---')
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "I enjoyed the show.", "I did not enjoy the show.")
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "I am happy.", "I am not happy.")

print('--- Subtle changes ---')
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "The movie was interesting.", "The movie was boring.")
counterfactual_analysis(nlp_model, nlp_tokenizer,
    "I recommend this.", "I do not recommend this.")

In [ ]:
# Systematic word importance: leave-one-out analysis
def leave_one_out_importance(model, tokenizer, text):
    """Measure each word's importance by removing it and observing the confidence change."""
    words = text.split()
    base_label, base_conf, base_probs = predict_sentiment(model, tokenizer, text)
    base_score = base_probs[1]  # positive probability
    
    importances = []
    for i, word in enumerate(words):
        modified = ' '.join(words[:i] + words[i+1:])
        _, _, mod_probs = predict_sentiment(model, tokenizer, modified)
        mod_score = mod_probs[1]
        importances.append(base_score - mod_score)  # positive = word pushes toward positive
    
    # Plot
    fig, ax = plt.subplots(figsize=(max(8, len(words) * 0.8), 3))
    colors = ['coral' if v < 0 else 'steelblue' for v in importances]
    ax.bar(range(len(words)), importances, color=colors, edgecolor='white')
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Importance\n(+ = pushes positive, - = pushes negative)')
    ax.set_title(f'Leave-One-Out Word Importance: "{text}" → {base_label} ({base_conf:.0%})', fontsize=10)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

leave_one_out_importance(nlp_model, nlp_tokenizer, "I absolutely loved this movie.")
leave_one_out_importance(nlp_model, nlp_tokenizer, "This was a complete waste of time.")
leave_one_out_importance(nlp_model, nlp_tokenizer, "The acting was excellent but the story was terrible.")

### 🤔 Questions

1. Which words have the greatest impact when swapped or removed?
2. Does the model handle negation correctly? ("I enjoyed" vs "I did not enjoy")
3. Does the leave-one-out importance align with the attention visualisation from Part 6? Where do they disagree?
4. Which method (attention or counterfactual) gives a more reliable explanation of model behaviour? Why?

*Double-click here to write your answers...*

---
# Part 9: Bias and Fairness

Explanations can reveal hidden biases in models. Let's probe whether the sentiment model associates names, locations, or demographics with sentiment.

In [ ]:
# Bias probes: does changing a name or demographic affect sentiment?
print('=== Bias Probes ===\n')

print('--- Name sensitivity ---')
for name in ['John', 'Aisha', 'Wei', 'Maria']:
    text = f"{name} gave an excellent presentation."
    label, conf, _ = predict_sentiment(nlp_model, nlp_tokenizer, text)
    print(f'  "{text}" → {label} ({conf:.1%})')

print('\n--- Location sensitivity ---')
for location in ['American', 'Chinese', 'Nigerian', 'French']:
    text = f"The {location} restaurant was wonderful."
    label, conf, _ = predict_sentiment(nlp_model, nlp_tokenizer, text)
    print(f'  "{text}" → {label} ({conf:.1%})')

### 🤔 Questions

1. Did changing names or nationalities affect the sentiment predictions? Should it?
2. Why might these biases emerge in pretrained models?
3. How can explanations help detect bias before deployment?
4. Should explainability be mandatory for high-risk AI applications? Why or why not?

*Double-click here to write your answers...*

---
# Part 10: Explainability Report — Mini Challenge 

Choose **one** of the following and write a short explainability report:

### Option A — Vision (Grad-CAM)
Upload your own image (or find one online), run Grad-CAM, and analyse the explanation.

### Option B — NLP (Attention + Counterfactuals)
Choose a sentence that the model gets wrong or is uncertain about, and investigate why.

### Your Report Must Include:

1. **Prediction:** What did the model predict? With what confidence?
2. **Explanation:** What evidence supports (or contradicts) the prediction? Show a Grad-CAM heatmap or attention/counterfactual analysis.
3. **Trust Assessment:** Do you trust this prediction? Why or why not?
4. **Failure Analysis:** What weaknesses did the explanation reveal?
5. **Improvement Suggestion:** How would you improve the model or the explanation method?

In [ ]:
# Your explainability report code here
# ...

---
# Part 11: Reflection

1. What is the difference between interpretability and explainability?
2. What does Grad-CAM tell us? What are its limitations?
3. What does attention visualisation tell us? Why might it be misleading?
4. Why are counterfactual explanations more rigorous than attention visualisation?
5. Can a highly accurate model still be dangerous? Give a concrete example.
6. Should AI systems always provide explanations for their predictions?
7. Looking back at the entire course (Weeks 3–12), how does explainability connect to the other topics — transfer learning, foundation models, representation learning?

*Double-click here to write your reflection...*

---
# Bonus (Optional)

### Vision
- Compare Grad-CAM across different target layers (early vs late)
- Try other explanation methods: `ScoreCAM`, `GradCAM++`, `EigenCAM` (all available in `pytorch_grad_cam`)
- Analyse adversarial examples: add small perturbations and see how explanations change

### NLP
- Investigate individual attention heads (not just the average)
- Test sarcasm: "Oh great, another meeting" — can the model detect sarcasm?
- Compare explanations from DistilBERT vs BERT-base

### General
- Explore SHAP (SHapley Additive exPlanations)
- Explore LIME (Local Interpretable Model-agnostic Explanations)
- Explore Integrated Gradients



---

## 🔑 Key Takeaway

Building a neural network is only the first step. Real-world AI systems must also be **understandable**, **trustworthy**, and **accountable**. Explainability techniques — Grad-CAM for vision, attention visualisation and counterfactuals for NLP — help us investigate what a model has learned, identify failure modes, uncover hidden biases, and make informed decisions about when a model should be trusted.

The best explanation is not always the most visual — **counterfactual analysis** ("what would change the prediction?") often reveals more about model behaviour than attention or saliency maps alone.